In [ ]:
# Cell 1: 安装依赖
!pip install diffusers transformers accelerate xformers pillow controlnet-aux huggingface_hub -q
print("✅ 依赖就绪")

In [ ]:
# Cell 2: 加载模型
import torch, gc
from diffusers import StableDiffusionXLPipeline, StableDiffusionXLControlNetPipeline, ControlNetModel
from controlnet_aux import CannyDetector

MODEL_ID = "cagliostrolab/animagine-xl-4.0"

print("加载 Animagine XL 4.0...")
pipe_sdxl = StableDiffusionXLPipeline.from_pretrained(MODEL_ID, torch_dtype=torch.float16, use_safetensors=True)

print("加载 ControlNet Canny...")
cn = ControlNetModel.from_pretrained("diffusers/controlnet-canny-sdxl-1.0", torch_dtype=torch.float16)

print("组装管线...")
pipe = StableDiffusionXLControlNetPipeline(
    vae=pipe_sdxl.vae, text_encoder=pipe_sdxl.text_encoder, text_encoder_2=pipe_sdxl.text_encoder_2,
    tokenizer=pipe_sdxl.tokenizer, tokenizer_2=pipe_sdxl.tokenizer_2,
    unet=pipe_sdxl.unet, controlnet=cn, scheduler=pipe_sdxl.scheduler,
)
pipe = pipe.to("cuda")
pipe.enable_xformers_memory_efficient_attention()

canny = CannyDetector()
del pipe_sdxl
gc.collect(); torch.cuda.empty_cache()
print("✅ 模型就绪")

In [ ]:
# Cell 3: 数据库 + 提示词模板 v1.0
from pathlib import Path
import json
from google.colab import drive
drive.mount('/content/drive')

DRIVE = Path("/content/drive/MyDrive")

with open(DRIVE / "all_cards_export.json", encoding="utf-8") as f:
    all_cards = json.load(f)
print(f"卡牌总数: {len(all_cards)}")

DYN = {"QH":"秦汉","SG":"三国","LJ":"两晋南北朝","ST":"隋唐","SY":"宋元","MQ":"明清"}

def get_dyn(c):
    pfx = c.get("card_id","").split("-")[0] if "-" in str(c.get("card_id")) else ""
    return DYN.get(pfx, c.get("dynasty",""))

def get_tags(c):
    t = c.get("tags",[])
    if isinstance(t, str):
        try: t = json.loads(t)
        except: t = [x.strip() for x in t.split(",")]
    return "、".join(t[:5]) if t else ""

def get_detail(c):
    for k in ["short_desc","story","knowledge_point"]:
        v = c.get(k)
        if v and len(str(v))>3: return str(v)[:20]
    return ""

TMPL = {
    "person":  "{name}，{dynasty}历史人物，{tags}，中国传统汉服，历史写实风格，国风插画，古代场景背景，{detail}",
    "event":   "{name}，{dynasty}历史事件，{tags}，古代战场，军队将士，历史画风，宏大场景，{detail}",
    "place":   "{name}，{dynasty}历史地标，{tags}，古城建筑，宫殿城墙，无人，国风山水，{detail}",
    "weapon":  "{name}，{dynasty}古代兵器，{tags}，金属冷兵器，深色丝绸背景，静物，无人，{detail}",
    "classic": "{name}，{dynasty}古籍，{tags}，竹简帛书，毛笔，无人，书房静物，{detail}",
    "book":    "{name}，{dynasty}古籍，{tags}，竹简帛书，毛笔，无人，书房静物，{detail}",
    "dynasty": "{name}，{dynasty}王朝象征，{tags}，龙旗玉玺，皇宫，长城，恢宏大气，{detail}",
}

def make_prompt(c):
    t = c.get("type","person")
    return TMPL.get(t, TMPL["person"]).format(
        name=c.get("name",""), dynasty=get_dyn(c), tags=get_tags(c), detail=get_detail(c))

NEG = ("anime, manga, cartoon, chibi, moe, kawaii, "
       "bishounen, bishoujo, big eyes, small nose, pointed chin, "
       "overly feminine, overly masculine, exaggerated muscles, "
       "3d, realistic photo, photograph, photorealistic, "
       "western, european, modern clothing, modern weapon, gun, "
       "nsfw, nude, text, watermark, signature, username, "
       "lowres, worst quality, ugly, deformed, blurry, bad anatomy")

print("✅ 提示词 v1.0 就绪")

In [ ]:
# Cell 4: 生成函数 v1.0
from PIL import Image, ImageDraw
import hashlib

def gen_one(card):
    cid = card["card_id"]
    ctype = card.get("type", "person")
    ref = Image.new("RGB", (832, 1216), (45, 40, 35))
    draw = ImageDraw.Draw(ref)
    if ctype in ("person","event","人物","事件"):
        draw.ellipse((340,150,492,310), outline=(120,100,80), width=5)
        draw.rectangle((312,310,520,750), outline=(120,100,80), width=5)
    elif ctype in ("place","地点"):
        draw.rectangle((80,380,380,950), outline=(120,100,80), width=5)
        draw.polygon([(60,380),(230,160),(400,380)], outline=(120,100,80), width=5)
    elif ctype in ("weapon","兵器","classic","book","典籍"):
        draw.line((416,120,416,980), fill=(120,100,80), width=6)
    elif ctype in ("dynasty","朝代"):
        draw.ellipse((200,250,632,500), outline=(120,100,80), width=5)
        draw.rectangle((150,550,682,900), outline=(120,100,80), width=5)
    edge = canny(ref, low_threshold=50, high_threshold=150)
    prompt = make_prompt(card)
    seed = int(hashlib.md5(cid.encode()).hexdigest()[:8], 16)
    generator = torch.Generator("cuda").manual_seed(seed)
    return pipe(prompt=prompt, negative_prompt=NEG, image=edge,
                controlnet_conditioning_scale=0.85, num_inference_steps=30,
                guidance_scale=6.5, width=832, height=1216, generator=generator).images[0]

print("✅ 生成函数就绪")

In [ ]:
# Cell 5: 批量生成（断点续传）
import time

OUT = DRIVE / "card_output_guofeng4"
OUT.mkdir(parents=True, exist_ok=True)
MAX_RETRY = 3

print(f"总数: {len(all_cards)} | 输出: Google Drive/card_output_guofeng4/")
print("=" * 50)

for idx, card in enumerate(all_cards):
    cid = card["card_id"]
    name = card.get("name", cid)
    ctype = card.get("type", "person")
    out_path = OUT / f"{cid}.png"
    if out_path.exists():
        continue
    print(f"[{idx+1}/{len(all_cards)}] {cid}: {name} ({ctype})")
    for attempt in range(1, MAX_RETRY + 1):
        try:
            img = gen_one(card)
            img.save(out_path)
            print("  ✅")
            gc.collect(); torch.cuda.empty_cache()
            break
        except Exception as e:
            if attempt < MAX_RETRY:
                print(f"  ⚠️ 重试 {attempt}: {str(e)[:60]}")
                time.sleep(5)
            else:
                print(f"  ❌ 失败: {str(e)[:60]}")

done = len(list(OUT.glob("*.png")))
print(f"\n🎉 {done}/{len(all_cards)} 完成 → Google Drive/card_output_guofeng4/")